# 04g — Fine-tuning **goals multitask** (v3)

**Objetivo:** `LossSpec.finetune_goals_multitask()` — W/D/L como tarea primaria,
goles por equipo (HomeGoalsHead + AwayGoalsHead) como auxiliares.

- Datos: `DATA_ROOT` = `MyDrive/d10sformer`
- Código/ckpt: `PROJECT_ROOT` = `MyDrive/d10sformer-v2`
- Pretrain: `checkpoints_v1/pretrain_5ep/best.pt`

**Por qué este enfoque:**
- La cabeza de resultado optimiza *directamente* log-loss (la métrica objetivo).
- Las dos cabezas de goles (6 clases c/u) enseñan estructura adicional con
  mucho menos desbalance que las 36 clases de marcador conjunto.
- En inferencia: W/D/L desde `result_logits`; distribución de marcadores desde
  `P(h) × P(a)` para el Monte Carlo del Mundial.


---
## 1. Setup

In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

DRIVE = Path('/content/drive/MyDrive')

def _find_project_root() -> Path:
    for cand in (
        DRIVE / 'd10sformer-v2' / 'd10sformer-v2',
        DRIVE / 'd10sformer-v2',
    ):
        if (cand / 'src' / 'data' / 'collator.py').is_file():
            return cand.resolve()
    raise FileNotFoundError('No encuentro src/data/collator.py en d10sformer-v2')

if IN_COLAB:
    PROJECT_ROOT = _find_project_root()
    DATA_ROOT = (DRIVE / 'd10sformer').resolve()
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_ROOT = PROJECT_ROOT

_src = str(PROJECT_ROOT / 'src')
sys.path[:] = [p for p in sys.path if Path(p).name != 'src' and 'd10sformer' not in p]
sys.path.insert(0, _src)

from paths import ensure_paths, print_paths
paths = ensure_paths(project_root=PROJECT_ROOT, data_root=DATA_ROOT)
print_paths(paths)

ROOT           = paths.project_root
DATA_PROCESSED = paths.data_processed
CORPUS_DIR     = paths.corpus_dir
VOCAB_PATH     = paths.vocab_path
CKPT_DIR       = paths.checkpoints
CHECKPOINTS_V1 = paths.checkpoints_v1
DATA_RAW       = paths.data_raw

# Auditoría
def _chk(label, p):
    return f'{"OK" if p.exists() else "MISSING":7}  {label}: {p}'

print('\n--- Auditoría 04g ---')
for label, p in [
    ('collator.py',      PROJECT_ROOT / 'src/data/collator.py'),
    ('trainer.py',       PROJECT_ROOT / 'src/training/trainer.py'),
    ('d10sformer.py',    PROJECT_ROOT / 'src/models/d10sformer.py'),
    ('vocab.json',       paths.vocab_path),
    ('finetune_train',   paths.corpus_dir / 'finetune_train.pkl'),
    ('pretrain best.pt', paths.checkpoints_v1 / 'pretrain_5ep/best.pt'),
]:
    print(_chk(label, p))

# Verificar que tenemos las nuevas cabezas
_tr = (PROJECT_ROOT / 'src/training/trainer.py').read_text()
assert 'finetune_goals_multitask' in _tr, 'trainer.py sin finetune_goals_multitask'
_co = (PROJECT_ROOT / 'src/data/collator.py').read_text()
assert 'home_goals_labels' in _co, 'collator.py sin home_goals_labels'
_mo = (PROJECT_ROOT / 'src/models/d10sformer.py').read_text()
assert 'home_goals_head' in _mo, 'd10sformer.py sin home_goals_head'
print('\n✓ Setup OK — goals multitask disponible')


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Necesitamos GPU. Runtime → Change runtime type → T4.')
print(f'torch: {torch.__version__}  |  GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from collections import Counter
import pandas as pd
from torch.utils.data import DataLoader

from data.dataset import MatchDataset
from models.d10sformer import D10Sformer, D10SformerConfig
from training.trainer import Trainer, TrainerConfig, LossSpec

PRETRAIN_CKPT = CHECKPOINTS_V1 / 'pretrain_5ep' / 'best.pt'
assert PRETRAIN_CKPT.exists(), f'No encuentro pretrain ckpt: {PRETRAIN_CKPT}'
print(f'✓ PRETRAIN_CKPT: {PRETRAIN_CKPT}')


---
## 2. Cargar corpus

In [ ]:
import pickle
from data.vocabulary import FootballVocab
from data.tokenizer import MatchTokenizer

vocab = FootballVocab.load(VOCAB_PATH)
tokenizer = MatchTokenizer(vocab, max_seq_length=80)

_pad_id = vocab.token_to_id.get('[PAD]', 0)
print(f'Vocab: {len(vocab):,} tokens  |  PAD id={_pad_id}')

with open(CORPUS_DIR / 'finetune_train.pkl', 'rb') as f:
    finetune_docs = pickle.load(f)
with open(CORPUS_DIR / 'val.pkl', 'rb') as f:
    val_docs = pickle.load(f)
with open(CORPUS_DIR / 'test.pkl', 'rb') as f:
    test_docs = pickle.load(f)

print(f'Finetune train: {len(finetune_docs):,}')
print(f'Val:            {len(val_docs):,}')
print(f'Test:           {len(test_docs):,}')


---
## 3. Datasets + DataLoaders

In [ ]:
from data.collator import MLMCollator, LabelMappedCollator

ds_train = MatchDataset(finetune_docs, tokenizer)
ds_val   = MatchDataset(val_docs,      tokenizer)
ds_test  = MatchDataset(test_docs,     tokenizer)

collator = LabelMappedCollator(MLMCollator(vocab, mlm_probability=0.15, seed=42))

# Sanity-check: nuevos campos goals presentes
_t = collator([ds_train[0]])
assert hasattr(_t, 'home_goals_labels'), 'CollatedBatch sin home_goals_labels — actualizá collator.py'
assert _t.score_labels.max() < 36
print(f'✓ CollatedBatch con goals labels: {_t.home_goals_labels}, {_t.away_goals_labels}')

BATCH_SIZE = 64
train_loader = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collator, num_workers=0)
val_loader   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=0)
test_loader  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collator, num_workers=0)
steps_per_epoch = len(train_loader)
print(f'Train batches/época: {steps_per_epoch}')


---
## 4. Distribución de clases y class weights

In [ ]:
# ── Result class weights (3 clases: home_win / draw / away_win) ──────
result_counts = Counter()
for sample in ds_train:
    local_id = collator.result_map.get(
        sample.target_result_id if sample.target_result_id is not None else -1, -1
    )
    if local_id >= 0:
        result_counts[local_id] += 1

n_res = sum(result_counts.values())
result_weights = [n_res / (3 * max(result_counts.get(i, 1), 1)) for i in range(3)]
mean_rw = sum(result_weights) / 3
result_weights = [w / mean_rw for w in result_weights]

labels_r = ['home_win', 'draw', 'away_win']
print('Result class weights:')
for i, (lbl, cnt, w) in enumerate(zip(labels_r,
    [result_counts.get(i,0) for i in range(3)], result_weights)):
    print(f'  [{i}] {lbl:12}: count={cnt:5d}  weight={w:.3f}')

# ── Goals class weights (6 clases: 0..5 goles) ───────────────────────────
home_counts = Counter()
away_counts = Counter()
for sample in ds_train:
    score_local = collator.score_map.get(
        sample.target_score_id if sample.target_score_id is not None else -1, -1
    )
    if score_local >= 0:
        home_counts[score_local // 6] += 1
        away_counts[score_local % 6]  += 1

def _goal_weights(counts):
    n = sum(counts.values())
    raw = [n / (6 * max(counts.get(i, 1), 1)) for i in range(6)]
    m = sum(raw) / 6
    return [w / m for w in raw]

home_goal_weights = _goal_weights(home_counts)
away_goal_weights = _goal_weights(away_counts)

print('\nHome goals class weights:')
for i in range(6):
    print(f'  {i} goles: count={home_counts.get(i,0):5d}  weight={home_goal_weights[i]:.3f}')
print('\nAway goals class weights:')
for i in range(6):
    print(f'  {i} goles: count={away_counts.get(i,0):5d}  weight={away_goal_weights[i]:.3f}')


---
## 5. Modelo

In [ ]:
model_config = D10SformerConfig(
    vocab_size=len(vocab),
    d_model=256, num_layers=6, num_heads=8, d_ff=1024,
    max_seq_length=80, num_segments=8,
    dropout=0.1, attention_dropout=0.1,
    pad_token_id=_pad_id,
    tie_mlm_weights=True,
)
model = D10Sformer(model_config)
print(model.parameter_breakdown())

# Cargamos desde pretrain; las cabezas de goles y resultado arrancan random
# (no estaban en pretrain), strict=False ignora claves nuevas/faltantes.
ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu', weights_only=False)
missing, unexpected = model.load_state_dict(ckpt['model_state_dict'], strict=False)
print(f'\n✓ Pretrain cargado (paso {ckpt["step"]}, val_loss={ckpt["best_val_loss"]:.4f})')
if missing:
    print(f'  Capas nuevas (random init): {missing}')


---
## 6. Configuración del Trainer

In [ ]:
EPOCHS    = 20
MAX_STEPS = steps_per_epoch * EPOCHS
print(f'EPOCHS={EPOCHS}  |  Pasos totales={MAX_STEPS}')

trainer_config = TrainerConfig(
    lr=5e-5, weight_decay=0.01, grad_clip_norm=1.0,
    warmup_ratio=0.1,
    max_steps=MAX_STEPS,
    mixed_precision=True,
    log_every=25,
    eval_every=100,
    save_every=500,
    save_best=True,
    output_dir=str(CKPT_DIR),
    run_name='finetune_goals_20ep',
    seed=42,
)

loss_spec = LossSpec.finetune_goals_multitask(
    lambda_mlm=0.1,
    lambda_result=1.0,
    lambda_home_goals=0.5,
    lambda_away_goals=0.5,
    result_class_weights=result_weights,
    home_goals_class_weights=home_goal_weights,
    away_goals_class_weights=away_goal_weights,
)
print(f'\nLossSpec: use_result={loss_spec.use_result}  use_goals={loss_spec.use_goals}')
print(f'          λ_result={loss_spec.lambda_result}  λ_home={loss_spec.lambda_home_goals}  λ_away={loss_spec.lambda_away_goals}  λ_mlm={loss_spec.lambda_mlm}')

trainer = Trainer(
    model=model, train_loader=train_loader, val_loader=val_loader,
    config=trainer_config, loss_spec=loss_spec,
)
print(f'\nDevice: {trainer.device}  |  AMP: {trainer.use_amp}')


---
## 7. Fine-tuning (20 épocas)

In [ ]:
import time
t0 = time.time()
trainer.train()
elapsed = time.time() - t0
print(f'\n✓ Fine-tuning completado en {elapsed/60:.1f} min ({MAX_STEPS/elapsed:.1f} step/s)')
print(f'  Best val_loss: {trainer.best_val_loss:.4f}')


---
## 8. Evaluación sobre VAL y TEST

In [ ]:
from eval.metrics import evaluate_all

best_path = trainer.output_dir / 'best.pt'
best_ckpt = torch.load(best_path, map_location=trainer.device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'], strict=False)
model.eval()
print(f'✓ Cargado best (step={best_ckpt["step"]}, val_loss={best_ckpt["best_val_loss"]:.4f})')

@torch.no_grad()
def predict_result_probs(loader):
    """Predice P(W/D/L) directamente desde result_logits (cabeza primaria)."""
    all_probs, all_true = [], []
    for batch in loader:
        batch = batch.to(trainer.device)
        out   = model(batch.token_ids, batch.segment_ids,
                      attention_mask=batch.attention_mask)
        probs  = F.softmax(out['result_logits'], dim=-1).cpu().numpy()  # (B, 3)
        labels = batch.result_labels.cpu().numpy()
        for p, y in zip(probs, labels):
            if y == -100: continue
            all_probs.append(p); all_true.append(int(y))
    return np.array(all_true), np.array(all_probs)

y_val,  p_val  = predict_result_probs(val_loader)
y_test, p_test = predict_result_probs(test_loader)

metrics_val  = evaluate_all(y_val,  p_val)
metrics_test = evaluate_all(y_test, p_test)
print('\n=== D10Sformer 4g (goals multitask) ===')
print(f'{"métrica":<20} {"VAL":<12} {"TEST":<12}')
for k in ['log_loss', 'brier', 'ece', 'accuracy']:
    print(f'{k:<20} {metrics_val[k]:<12.4f} {metrics_test[k]:<12.4f}')


In [ ]:
# Confusion matrix y distribución de draws
preds_val = p_val.argmax(axis=1)
cm = np.zeros((3, 3), dtype=int)
for yt, yp in zip(y_val, preds_val):
    cm[yt, yp] += 1

labels_cm = ['home_win', 'draw', 'away_win']
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() * 0.6 else 'black')
ax.set_xticks(range(3)); ax.set_xticklabels(labels_cm)
ax.set_yticks(range(3)); ax.set_yticklabels(labels_cm)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title(f'Confusion matrix VAL (n={len(y_val)}) — 04g goals multitask')
plt.colorbar(im); plt.tight_layout(); plt.show()

print(f'\nDistribución VAL real: {Counter(y_val.tolist())}')
print(f'Predicciones VAL:      {Counter(preds_val.tolist())}')
print(f'Draws predichos: {(preds_val==1).sum()} / {len(y_val)} ({100*(preds_val==1).mean():.1f}%)')


---
## 9. Distribución de marcadores (para Monte Carlo)

In [ ]:
# Las cabezas de goles permiten estimar P(h-a) para el simulador del Mundial.
# P(score h-a) = P(home=h) × P(away=a)  [aproximación Dixon-Coles]

@torch.no_grad()
def predict_score_distribution(loader):
    """Devuelve P(home_goals), P(away_goals) para cada partido."""
    all_ph, all_pa, all_true_h, all_true_a = [], [], [], []
    for batch in loader:
        batch = batch.to(trainer.device)
        out   = model(batch.token_ids, batch.segment_ids,
                      attention_mask=batch.attention_mask)
        ph = F.softmax(out['home_goals_logits'], dim=-1).cpu().numpy()  # (B, 6)
        pa = F.softmax(out['away_goals_logits'], dim=-1).cpu().numpy()  # (B, 6)
        lh = batch.home_goals_labels.cpu().numpy()
        la = batch.away_goals_labels.cpu().numpy()
        for i in range(len(ph)):
            if lh[i] == -100: continue
            all_ph.append(ph[i]); all_pa.append(pa[i])
            all_true_h.append(int(lh[i])); all_true_a.append(int(la[i]))
    return (np.array(all_ph), np.array(all_pa),
            np.array(all_true_h), np.array(all_true_a))

ph_val, pa_val, th_val, ta_val = predict_score_distribution(val_loader)

# MAE de goles esperados
exp_home = (ph_val * np.arange(6)).sum(axis=1)
exp_away = (pa_val * np.arange(6)).sum(axis=1)
mae_h = np.abs(exp_home - th_val).mean()
mae_a = np.abs(exp_away - ta_val).mean()
print(f'MAE goles locales  (VAL): {mae_h:.3f}')
print(f'MAE goles visitante(VAL): {mae_a:.3f}')

# Distribución media de goles predichos
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, probs, true, title in [
    (axes[0], ph_val, th_val, 'Goles locales (VAL)'),
    (axes[1], pa_val, ta_val, 'Goles visitante (VAL)'),
]:
    pred_dist = probs.mean(axis=0)
    true_hist = np.bincount(true, minlength=6) / len(true)
    x = np.arange(6)
    ax.bar(x - 0.2, true_hist, 0.4, label='Real', alpha=0.7)
    ax.bar(x + 0.2, pred_dist, 0.4, label='Predicho', alpha=0.7)
    ax.set_xticks(x); ax.set_xlabel('Goles'); ax.set_ylabel('Proporción')
    ax.set_title(title); ax.legend()
plt.tight_layout(); plt.show()


---
## 10. Comparación final vs baselines

In [ ]:
baselines = {
    'LogReg':                           {'log_loss': 0.8610, 'brier': 0.5071, 'ece': 0.0236, 'accuracy': 0.6005},
    'XGBoost':                          {'log_loss': 0.8663, 'brier': 0.5105, 'ece': 0.0182, 'accuracy': 0.6026},
    'LightGBM':                         {'log_loss': 0.8744, 'brier': 0.5135, 'ece': 0.0250, 'accuracy': 0.5990},
    'D10Sformer 4d (sin weights)':       {'log_loss': 0.8846, 'brier': 0.5221, 'ece': 0.0400, 'accuracy': 0.5912},
    'D10Sformer 4f (score-primary)':     {'log_loss': 0.8902, 'brier': 0.5226, 'ece': 0.0334, 'accuracy': 0.5961},
    'D10Sformer 4g (goals-mt, val)':     {k: metrics_val[k]  for k in ['log_loss','brier','ece','accuracy']},
    'D10Sformer 4g (goals-mt, test)':    {k: metrics_test[k] for k in ['log_loss','brier','ece','accuracy']},
}

df = pd.DataFrame(baselines).T
print(df.to_string(float_format=lambda x: f'{x:.4f}'))

best_ll  = min(baselines[k]['log_loss'] for k in ['LogReg','XGBoost','LightGBM'])
best_acc = max(baselines[k]['accuracy'] for k in ['LogReg','XGBoost','LightGBM'])
best_ece = min(baselines[k]['ece']      for k in ['LogReg','XGBoost','LightGBM'])

def cmp(v, ref, lower_better=True):
    if lower_better:
        return f'{"✓ GANA" if v < ref else "✗ pierde"} (Δ={v-ref:+.4f})'
    return f'{"✓ GANA" if v > ref else "✗ pierde"} (Δ={v-ref:+.4f})'

print(f'\n--- D10Sformer 4g (test) vs mejor baseline ---')
print(f'log_loss: {metrics_test["log_loss"]:.4f} vs {best_ll:.4f} → {cmp(metrics_test["log_loss"], best_ll)}')
print(f'accuracy: {metrics_test["accuracy"]:.4f} vs {best_acc:.4f} → {cmp(metrics_test["accuracy"], best_acc, False)}')
print(f'ece:      {metrics_test["ece"]:.4f} vs {best_ece:.4f} → {cmp(metrics_test["ece"], best_ece)}')


---
## 11. Conclusiones de Fase 4g

Llenar al final:

- [ ] Draws predichos VAL: _____ de 293 reales
- [ ] D10Sformer 4g test log_loss: _____  vs LogReg 0.8610
- [ ] D10Sformer 4g test accuracy: _____  vs XGBoost 0.6026
- [ ] D10Sformer 4g test ECE: _____      vs XGBoost 0.0182
- [ ] MAE goles locales: _____
- [ ] MAE goles visitante: _____
- [ ] ¿La confusion matrix muestra draws? _____

**Checklist de mejoras aplicadas vs 4f:**
- [x] Result head como tarea primaria (optimiza log-loss directamente)
- [x] Class weights en result head (fix de empates)
- [x] Goals heads (6 clases c/u) como auxiliares con class weights
- [x] 20 épocas en lugar de 15
- [x] λ_mlm reducido a 0.1

**Próximo paso:**
- Si ganamos en log_loss → Fase 5 (eval profunda + reliability diagrams) + Fase 6 (Monte Carlo).
- Si aún perdemos pero Δ < 0.01 → más datos (football-data.co.uk + Wyscout) y re-pretrain.
